# 01 – Data Discovery & Exploration



Wetter-/Score-Datensatz für das Data-Mining-Finale.



**Dual-Environment-Strategie**

| Umgebung | Trainingsdaten | Hinweis |

|----------|----------------|--------|

| **Lokal (schwacher PC)** | `train_sample.csv` (10.000 Zeilen) | Vorher: `python scripts/create_sample.py` |

| **Google Colab** | `train.csv` (~1,1 GB) | Drive mounten, Pfade unten anpassen |



`test.csv` ist lokal klein genug (~19 MB) und wird in beiden Umgebungen voll geladen.



> **Wichtig:** Ergebnisse aus dem lokalen Sample sind **indikativ**, nicht repräsentativ für das Gesamtdataset. Finale Aussagen nur nach Colab-Lauf mit vollem `train.csv`.



In [ ]:
import os

import sys
from pathlib import Path

# Repo root (works when notebook runs from /notebooks or project root)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists() and (PROJECT_ROOT.parent / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import warnings

from collections import defaultdict



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (12, 6)



RANDOM_STATE = 42



## 1. Umgebung erkennen & Pfade setzen



Passe in Colab `COLAB_DATA_DIR` an deinen Google-Drive-Ordner an.



In [ ]:
try:

    from google.colab import drive

    IS_COLAB = True

except ImportError:

    IS_COLAB = False



# --- Projektroot finden (lokal + Colab) ---

def find_project_root() -> Path:

    candidates = [Path.cwd(), Path.cwd().parent]

    if IS_COLAB:

        candidates += [

            Path("/content/DataMining_Final-Project"),

            Path("/content/drive/MyDrive/DataMining/DataMining_Final-Project"),

            Path("/content/drive/MyDrive/DataMining_Final-Project"),

        ]

    for p in candidates:

        if (p / "config" / "paths.py").exists():

            return p.resolve()

    return Path.cwd().resolve()



if IS_COLAB:

    drive.mount("/content/drive")



PROJECT_ROOT = find_project_root()

if IS_COLAB and not (PROJECT_ROOT / "config").exists():

  # Nach Mount: nochmal suchen

    PROJECT_ROOT = find_project_root()



sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)



# config importieren oder Fallback (nur Notebook hochgeladen)

try:

    from config.paths import setup_environment, COLAB_DATA_DIR, FIGURES_DIR

    env = setup_environment()

except ModuleNotFoundError:

    print("⚠ config/ nicht gefunden — Fallback-Pfade (nur Daten auf Drive nötig).")

    print("   Tipp: Gesamtes Repo nach Drive legen, z.B. MyDrive/DataMining/DataMining_Final-Project/")

    COLAB_DATA_DIR = "/content/drive/MyDrive/DataMining/data"

    FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    data_dir = Path(COLAB_DATA_DIR)

    env = {

        "is_colab": IS_COLAB,

        "project_root": PROJECT_ROOT,

        "data_dir": data_dir,

        "train_path": data_dir / "train.csv",

        "test_path": data_dir / "test.csv",

        "figures_dir": FIGURES_DIR,

        "use_chunked_train": IS_COLAB,

        "chunk_size": 500_000,

        "env_label": "Colab (Fallback-Pfade)" if IS_COLAB else "Lokal",

    }



TRAIN_PATH = str(env["train_path"])

TEST_PATH = str(env["test_path"])

USE_CHUNKED_TRAIN = env["use_chunked_train"]

CHUNK_SIZE = env["chunk_size"]

ENV_LABEL = env["env_label"]

FIGURES_DIR = env["figures_dir"]



print(f"Umgebung: {ENV_LABEL}")

print(f"  PROJECT_ROOT: {PROJECT_ROOT}")

print(f"  config OK:    {(PROJECT_ROOT / 'config' / 'paths.py').exists()}")

print(f"  TRAIN_PATH: {TRAIN_PATH}")

print(f"  TEST_PATH:  {TEST_PATH}")

print(f"  FIGURES:    {FIGURES_DIR}")

print(f"  Chunked:    {USE_CHUNKED_TRAIN}")



for path, name in [(TRAIN_PATH, "Train"), (TEST_PATH, "Test")]:

    if os.path.exists(path):

        size_mb = os.path.getsize(path) / 1e6

        print(f"  ✓ {name} gefunden ({size_mb:.1f} MB)")

    else:

        print(f"  ✗ {name} FEHLT: {path}")

        if IS_COLAB:

            print("     → train.csv & test.csv nach MyDrive/DataMining/data/")

        else:

            print("     → Lokal: python scripts/create_sample.py")



## 2. Hilfsfunktionen



**Datums-Workaround:** `pd.to_datetime()` scheitert ab Jahr 2262. Wir splitten `YYYY-MM-DD` in `year`, `month`, `day`.



In [ ]:
def parse_date_columns(df: pd.DataFrame) -> pd.DataFrame:

    """Split date string into year, month, day (no pd.to_datetime)."""

    parts = df["date"].astype(str).str.split("-", expand=True)

    df = df.copy()

    df["year"] = parts[0].astype(int)

    df["month"] = parts[1].astype(int)

    df["day"] = parts[2].astype(int)

    return df





def load_train(path: str, chunked: bool = False, chunk_size: int = 500_000) -> pd.DataFrame:

    """Load train.csv (full or sample). Use chunked=False for local sample."""

    if chunked:

        chunks = []

        for chunk in pd.read_csv(path, chunksize=chunk_size):

            chunks.append(chunk)

        return pd.concat(chunks, ignore_index=True)

    return pd.read_csv(path)





def days_in_month(month: int) -> int:

    """Approximate day-of-year helper (synthetic calendar, no leap years)."""

    days_before = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]

    return days_before[month - 1]



## 3. Daten laden



- **Lokal:** `train_sample.csv` direkt in den RAM.

- **Colab:** optional chunkweise (kann einige Minuten dauern).



In [ ]:
df = load_train(TRAIN_PATH, chunked=USE_CHUNKED_TRAIN, chunk_size=CHUNK_SIZE or 500_000)

df_test = pd.read_csv(TEST_PATH)



df = parse_date_columns(df)

df_test = parse_date_columns(df_test)



df = df.sort_values(["region_id", "year", "month", "day", "date"]).reset_index(drop=True)

df_test = df_test.sort_values(["region_id", "year", "month", "day", "date"]).reset_index(drop=True)



print(f"Train: {len(df):,} Zeilen × {df.shape[1]} Spalten")

print(f"Test:  {len(df_test):,} Zeilen × {df_test.shape[1]} Spalten")

if not IS_COLAB:

    print("\n⚠ Lokal nur Sample – Regionen/Zeiträume können unvollständig sein!")

df.head()



## 4. Struktur & Spaltentypen


In [ ]:
print("=== Train – Info ===")

df.info()



print("\n=== Spalten ===")

feature_cols = [c for c in df.columns if c not in ("region_id", "date", "score")]

print(f"Features ({len(feature_cols)}): {feature_cols}")

print(f"Target: score")

print(f"IDs:   region_id, date")



print("\n=== Test – fehlende Spalten vs. Train ===")

train_cols = set(df.columns)

test_cols = set(df_test.columns)

print(f"Nur in Train: {train_cols - test_cols}")

print(f"Nur in Test:  {test_cols - train_cols}")



display(df.describe().T)

display(df_test.describe().T)



## 5. Fehlende Werte & Zielvariable `score`


In [ ]:
print("=== Fehlende Werte (Train) ===")



missing = df.isnull().sum()



missing_pct = (missing / len(df) * 100).round(2)



missing_df = pd.DataFrame({"count": missing, "pct": missing_pct})



display(missing_df[missing_df["count"] > 0].sort_values("count", ascending=False))







n_scored = df["score"].notna().sum()



print(f"\nZeilen mit Score: {n_scored:,} / {len(df):,} ({100 * n_scored / len(df):.2f}%)")







df_scored = df[df["score"].notna()].copy()



print("\n=== Score-Verteilung (nur gelabelte Zeilen) ===")



display(df_scored["score"].describe())



print(f"Anteil Score == 0: {(df_scored['score'] == 0).mean() * 100:.1f}%")







fig, axes = plt.subplots(1, 2, figsize=(14, 5))



df_scored["score"].hist(bins=30, ax=axes[0], edgecolor="black")



axes[0].set_title("Histogramm: score")



axes[0].set_xlabel("score")







df_scored["score"].value_counts().sort_index().plot(kind="bar", ax=axes[1])



axes[1].set_title("Häufigkeit pro Score-Wert")



plt.tight_layout()



plt.savefig(FIGURES_DIR / "score_distribution.png", dpi=120, bbox_inches="tight")



plt.show()





## 6. Zeitliche Analyse



- Datumsbereich pro Region

- Tägliche Beobachtungsfrequenz (Lücken in der Datumssequenz)

- Rhythmus der Score-Meldungen



In [ ]:
print("=== Globaler Datumsbereich ===")

print(f"Min: {df['date'].min()}  |  Max: {df['date'].max()}")

print(f"Jahre: {df['year'].min()} – {df['year'].max()}")



print("\n=== Pro Region: Zeilen, Datumsbereich, Score-Rate ===")

region_summary = (

    df.groupby("region_id")

    .agg(

        n_rows=("date", "count"),

        date_min=("date", "min"),

        date_max=("date", "max"),

        n_scored=("score", lambda s: s.notna().sum()),

    )

    .assign(score_rate=lambda x: (x["n_scored"] / x["n_rows"] * 100).round(2))

)

display(region_summary)



fig, ax = plt.subplots(figsize=(10, 5))

region_summary["score_rate"].plot(kind="bar", ax=ax, color="steelblue")

ax.set_ylabel("Score vorhanden (%)")

ax.set_title("Anteil gelabelter Zeilen pro Region")

ax.set_xlabel("region_id")

plt.tight_layout()

plt.show()



In [ ]:
def check_date_gaps(region_df: pd.DataFrame, region: str) -> dict:

    """Check consecutive-day gaps using year/month/day ordering."""

    r = region_df.sort_values(["year", "month", "day"]).copy()

    r["ordinal"] = r["year"] * 372 + r["month"] * 31 + r["day"]  # synthetic ordinal

    diffs = r["ordinal"].diff().dropna()

    gap_days = (diffs[diffs > 1] - 1).astype(int)

    return {

        "region": region,

        "n_days": len(r),

        "n_gaps": int((diffs > 1).sum()),

        "max_gap": int(gap_days.max()) if len(gap_days) else 0,

        "mean_gap_when_gap": float(gap_days.mean()) if len(gap_days) else 0,

    }





gap_results = [check_date_gaps(df[df["region_id"] == r], r) for r in df["region_id"].unique()]

gap_df = pd.DataFrame(gap_results)

print("=== Fehlende Kalendertage (synthetische Ordinal-Methode) ===")

display(gap_df)



if not IS_COLAB:

    print("⚠ Im lokalen Sample sind oft nur wenige Regionen/Tage sichtbar.")



In [ ]:
def score_reporting_gaps(region_df: pd.DataFrame) -> pd.Series:

    """Days between consecutive score observations within a region."""

    scored = region_df[region_df["score"].notna()].sort_values(["year", "month", "day"])

    if len(scored) < 2:

        return pd.Series(dtype=float)

    ord_ = scored["year"] * 372 + scored["month"] * 31 + scored["day"]

    return ord_.diff().dropna()





all_gaps = []

for region in df["region_id"].unique():

    gaps = score_reporting_gaps(df[df["region_id"] == region])

    if len(gaps):

        all_gaps.append(gaps.rename(region))



if all_gaps:

    gap_series = pd.concat(all_gaps)

    print("=== Abstand zwischen Score-Meldungen (Tage) ===")

    print(gap_series.describe())

    print("\nHäufigste Abstände:")

    print(gap_series.value_counts().head(10))



    fig, ax = plt.subplots()

    gap_series.hist(bins=50, ax=ax, edgecolor="black")

    ax.set_title("Tage zwischen aufeinanderfolgenden Score-Meldungen")

    ax.set_xlabel("Tage")

    plt.tight_layout()

    plt.show()

else:

    print("Zu wenige Score-Meldungen für Gap-Analyse.")



## 7. Regionale Analyse



Vergleich der Wetter- und Score-Verteilungen über `region_id`.



In [ ]:
region_means = df.groupby("region_id")[["tmp", "prec", "humidity", "wind", "surf_pre"]].mean()



region_score = df_scored.groupby("region_id")["score"].agg(["mean", "std", "max", "count"])







print("=== Mittlere Wetterwerte pro Region ===")



display(region_means)



print("\n=== Score-Statistik pro Region (nur gelabelt) ===")



display(region_score)







fig, axes = plt.subplots(2, 2, figsize=(14, 10))



for ax, col in zip(axes.ravel(), ["tmp", "prec", "wind", "humidity"]):



    df.boxplot(column=col, by="region_id", ax=ax)



    ax.set_title(col)



    plt.suptitle("")



plt.tight_layout()



plt.savefig(FIGURES_DIR / "regional_weather_boxplots.png", dpi=120, bbox_inches="tight")



plt.show()





## 8. Feature-Verteilungen & Ausreißer


In [ ]:
weather_features = [



    "prec", "surf_pre", "humidity", "tmp", "dp_tmp", "wb_tmp",



    "tmp_max", "tmp_min", "tmp_range", "surf_tmp",



    "wind", "wind_max", "wind_min", "wind_range",



]







fig, axes = plt.subplots(4, 4, figsize=(16, 14))



for ax, col in zip(axes.ravel(), weather_features):



    df[col].hist(bins=40, ax=ax, edgecolor="black", alpha=0.7)



    ax.set_title(col)



plt.suptitle("Verteilungen aller Wetter-Features (Train)", y=1.01)



plt.tight_layout()



plt.savefig(FIGURES_DIR / "feature_distributions.png", dpi=120, bbox_inches="tight")



plt.show()







print("=== Extreme Werte (99.9 % Quantil) ===")



q999 = df[weather_features].quantile(0.999)



display(q999.to_frame("q99.9"))





## 9. Korrelationsanalyse


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()



corr = df[num_cols].corr()







fig, ax = plt.subplots(figsize=(14, 12))



sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, linewidths=0.1)



ax.set_title("Korrelationsmatrix (alle numerischen Spalten)")



plt.tight_layout()



plt.savefig(FIGURES_DIR / "feature_correlation.png", dpi=120, bbox_inches="tight")



plt.show()







if "score" in corr.columns:



    score_corr = corr["score"].drop("score").sort_values(key=abs, ascending=False)



    print("=== Korrelation mit score (absteigend |r|) ===")



    display(score_corr.to_frame("corr_with_score"))





In [ ]:
# Scatter: stärkste Korrelaten vs. score (nur gelabelte Zeilen)

if len(df_scored) > 10 and "score" in corr.columns:

    top_features = corr["score"].drop(["score", "year", "month", "day"], errors="ignore")

    top_features = top_features.abs().sort_values(ascending=False).head(6).index.tolist()



    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    for ax, feat in zip(axes.ravel(), top_features):

        ax.scatter(df_scored[feat], df_scored["score"], alpha=0.3, s=8)

        ax.set_xlabel(feat)

        ax.set_ylabel("score")

    plt.suptitle("Top-Features vs. score")

    plt.tight_layout()

    plt.show()



## 10. Saisonalität (Monat & Tag-im-Jahr)


In [ ]:
df["doy"] = df.apply(lambda r: days_in_month(r["month"]) + r["day"], axis=1)







monthly = df.groupby("month").agg(



    tmp_mean=("tmp", "mean"),



    prec_mean=("prec", "mean"),



    wind_mean=("wind", "mean"),



)



monthly_score = df_scored.groupby("month")["score"].mean()







fig, ax1 = plt.subplots(figsize=(12, 5))



ax1.plot(monthly.index, monthly["tmp_mean"], "o-", color="tab:red", label="Ø Temperatur")



ax1.set_ylabel("Ø tmp (°C)", color="tab:red")



ax1.set_xlabel("Monat")



ax1.set_xticks(range(1, 13))







ax2 = ax1.twinx()



ax2.bar(monthly_score.index, monthly_score.values, alpha=0.35, color="tab:blue", label="Ø Score")



ax2.set_ylabel("Ø score", color="tab:blue")



ax1.set_title("Saisonalität: Temperatur vs. durchschnittlicher Score")



plt.tight_layout()



plt.savefig(FIGURES_DIR / "seasonality_month.png", dpi=120, bbox_inches="tight")



plt.show()







print("=== Score-Meldungen pro Monat ===")



print(df_scored["month"].value_counts().sort_index())





In [ ]:
# Wochentag nicht verfügbar (synthetischer Kalender) → Score-Häufigkeit nach Tag-im-Monat

fig, ax = plt.subplots(figsize=(12, 4))

df_scored.groupby("day")["score"].count().plot(kind="bar", ax=ax, color="teal")

ax.set_title("Anzahl Score-Meldungen nach Tag-im-Monat (1–31)")

ax.set_xlabel("day")

plt.tight_layout()

plt.show()



## 11. Zeitreihen-Plot (eine Region)



Visueller Check: Wetter vs. Score über die Zeit.



In [ ]:
sample_region = df["region_id"].value_counts().index[0]



region_data = df[df["region_id"] == sample_region].copy()







fig, ax1 = plt.subplots(figsize=(14, 5))



ax1.plot(range(len(region_data)), region_data["tmp"], color="tab:red", alpha=0.6, label="tmp")



ax1.set_ylabel("Temperatur", color="tab:red")



ax1.set_xlabel("Index (chronologisch sortiert)")







ax2 = ax1.twinx()



scored_idx = region_data["score"].notna()



ax2.scatter(



    np.where(scored_idx)[0],



    region_data.loc[scored_idx, "score"],



    color="tab:blue",



    s=15,



    alpha=0.8,



    label="score",



)



ax2.set_ylabel("score", color="tab:blue")



plt.title(f"Region {sample_region}: Temperatur & Score")



plt.tight_layout()



plt.savefig(FIGURES_DIR / "timeseries_sample_region.png", dpi=120, bbox_inches="tight")



plt.show()





## 12. Lag-Analyse (Vorbereitung Feature Engineering)



Vergleich: aktuelles Wetter vs. verzögertes Wetter (7 / 14 / 21 Tage) gegenüber `score`.



In [ ]:
LAG_DAYS = [1, 3, 7, 14, 21]

LAG_FEATURES = ["tmp", "tmp_range", "prec", "wind", "surf_pre", "humidity"]



df_lag = df.copy()

for lag in LAG_DAYS:

    for feat in LAG_FEATURES:

        df_lag[f"{feat}_lag_{lag}"] = df_lag.groupby("region_id")[feat].shift(lag)



lag_cols = [c for c in df_lag.columns if "_lag_" in c]

lag_corr = df_lag[lag_cols + ["score"]].corr()["score"].drop("score").sort_values(key=abs, ascending=False)



print("=== Top Lag-Korrelationen mit score ===")

display(lag_corr.head(20).to_frame("corr"))



# Vergleich: gleiches Feature, verschiedene Lags

compare_rows = []

for feat in LAG_FEATURES:

    row = {"feature": feat, "same_day": df[[feat, "score"]].corr().iloc[0, 1]}

    for lag in LAG_DAYS:

        col = f"{feat}_lag_{lag}"

        row[f"lag_{lag}"] = df_lag[[col, "score"]].corr().iloc[0, 1]

    compare_rows.append(row)



lag_compare = pd.DataFrame(compare_rows).set_index("feature")

print("\n=== Korrelation same_day vs. Lags ===")

display(lag_compare)



## 13. Rolling-Window-Statistiken (Sneak Peek)


In [ ]:
WINDOW = 7

ROLL_FEATURES = ["tmp", "prec", "wind"]



df_roll = df.copy()

for feat in ROLL_FEATURES:

    g = df_roll.groupby("region_id")[feat]

    df_roll[f"{feat}_roll{WINDOW}_mean"] = g.transform(lambda s: s.rolling(WINDOW, min_periods=1).mean())

    df_roll[f"{feat}_roll{WINDOW}_std"] = g.transform(lambda s: s.rolling(WINDOW, min_periods=1).std())



roll_cols = [c for c in df_roll.columns if f"roll{WINDOW}" in c]

roll_corr = df_roll[roll_cols + ["score"]].corr()["score"].drop("score").sort_values(key=abs, ascending=False)

print(f"=== Rolling-{WINDOW}-Tage Korrelation mit score ===")

display(roll_corr.to_frame("corr"))



## 14. Extreme Wetterereignisse vs. Score


In [ ]:
THRESHOLDS = {

    "prec": df["prec"].quantile(0.95),

    "tmp_max": df["tmp_max"].quantile(0.95),

    "wind_max": df["wind_max"].quantile(0.95),

    "tmp_range": df["tmp_range"].quantile(0.95),

}



print("95%-Schwellen:", {k: round(v, 2) for k, v in THRESHOLDS.items()})



rows = []

for feat, thr in THRESHOLDS.items():

    sub = df_scored.copy()

    is_extreme = sub[feat] >= thr

    rows.append({

        "feature": feat,

        "threshold": round(thr, 2),

        "mean_score_extreme": sub.loc[is_extreme, "score"].mean(),

        "mean_score_normal": sub.loc[~is_extreme, "score"].mean(),

        "n_extreme": int(is_extreme.sum()),

        "n_normal": int((~is_extreme).sum()),

    })

display(pd.DataFrame(rows))



## 15. Test-Set Exploration



`test.csv` ist in beiden Umgebungen vollständig ladbar.



In [ ]:
print(f"Test: {len(df_test):,} Zeilen")

print(f"Spalten: {list(df_test.columns)}")

print(f"Datumsbereich: {df_test['date'].min()} – {df_test['date'].max()}")

print(f"Regionen: {sorted(df_test['region_id'].unique())}")



print("\n=== Test vs. Train – Feature-Ranges ===")

compare = []

for col in weather_features:

    compare.append({

        "feature": col,

        "train_min": df[col].min(),

        "train_max": df[col].max(),

        "test_min": df_test[col].min(),

        "test_max": df_test[col].max(),

    })

display(pd.DataFrame(compare))



if "score" in df_test.columns:

    print(f"Test hat score-Spalte: {df_test['score'].notna().sum()} Werte")

else:

    print("Test hat keine score-Spalte (Prediction-Set).")



## 16. Colab-only: Chunk-Summary über volles `train.csv`



**Nur in Google Colab ausführen.** Liest `train.csv` chunkweise, ohne alles in den RAM zu laden.

Überspringe diese Zelle lokal (oder `RUN_CHUNK_SUMMARY = False`).



In [ ]:
RUN_CHUNK_SUMMARY = IS_COLAB  # lokal auf False lassen







if not RUN_CHUNK_SUMMARY:



    print("Chunk-Summary übersprungen (lokal oder RUN_CHUNK_SUMMARY=False).")



else:



    FULL_TRAIN_PATH = str(env["data_dir"] / "train.csv")



    n_rows = 0



    score_count = 0



    null_counts = defaultdict(int)



    region_counts = defaultdict(int)



    year_counts = defaultdict(int)



    min_date = max_date = None







    for chunk in pd.read_csv(FULL_TRAIN_PATH, chunksize=CHUNK_SIZE):



        n_rows += len(chunk)



        score_count += chunk["score"].notna().sum()



        for col in chunk.columns:



            null_counts[col] += chunk[col].isna().sum()



        for r, c in chunk["region_id"].value_counts().items():



            region_counts[r] += c



        parts = chunk["date"].astype(str).str.split("-", expand=True)



        for y, c in parts[0].astype(int).value_counts().items():



            year_counts[y] += c



        dmin, dmax = chunk["date"].min(), chunk["date"].max()



        min_date = dmin if min_date is None or dmin < min_date else min_date



        max_date = dmax if max_date is None or dmax > max_date else max_date







    print(f"=== Voller Trainingsdatensatz (chunkweise) ===")



    print(f"Zeilen gesamt:     {n_rows:,}")



    print(f"Mit Score:         {score_count:,} ({100 * score_count / n_rows:.2f}%)")



    print(f"Datumsbereich:     {min_date} – {max_date}")



    print(f"Regionen:          {dict(sorted(region_counts.items()))}")



    print(f"Fehlende score:    {null_counts.get('score', 0):,}")



    print(f"Top-5 Jahre:       {dict(sorted(year_counts.items(), key=lambda x: -x[1])[:5])}")





## 17. EDA-Zusammenfassung (manuell ausfüllen nach dem Lauf)



Trage hier deine Erkenntnisse ein – zuerst lokal (Sample), dann in Colab validieren:



| Thema | Lokal (Sample) | Colab (voll) |

|-------|----------------|--------------|

| Zeilen / Score-Rate | | |

| Regionen & Datumsbereich | | |

| Fehlende Kalendertage | | |

| Score-Rhythmus (Tage zwischen Meldungen) | | |

| Stärkste Korrelaten mit score | | |

| Beste Lag-Features | | |

| Saisonalität (Monat) | | |

| Train vs. Test Distribution Shift | | |



**Nächste Schritte (laut docs/03_PROJECT_PLAN.md):**

1. Imputation fehlender Wetterwerte

2. Lag- & Rolling-Features finalisieren

3. `region_id`-Encoding

4. Baseline-Modell + Time-Series-CV in Colab

